# Lab Tasks - Solutions

This task involves constructing an ingredient co-occurrence network, often called a **recipe network**, from a dataset of recipes from the website [yummly.com](http://yummly.com) which covers a number of different cuisines, each associated with a specific region. Recipe networks can reveal interesting patterns about how ingredients are combined across different culinary traditions.

In [ ]:
from collections import Counter
import json, itertools
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## Task 1

Load the file *recipes.json*, and then calculate: 
1. The number of recipes per cuisine
2. The total number of unique ingredients
3. The most common ingredients across all recipes

In [ ]:
# load the data
fin = open("recipes.json","r")
data = json.load(fin)
fin.close()
print(f"Read {len(data)} recipes")

In [ ]:
# find the number of recipes per cuisine
cuisine_counts = Counter()
for recipe in data:
    cuisine_counts[recipe["cuisine"]] += 1
for cuisine, count in cuisine_counts.most_common():
    print(f"{cuisine}: {count}")

In [ ]:
# find total number of unique ingredients
all_ingredients = set()
for recipe in data:
    for ingredient in recipe["ingredients"]:
        all_ingredients.add(ingredient)
print(f"Total of {len(all_ingredients)} unique ingredients")

In [ ]:
# find the most common ingredients across all recipes
ingredient_counts = Counter()
for recipe in data:
    for ingredient in recipe["ingredients"]:
        ingredient_counts[ingredient] += 1
# display the top 30
rank = 0
for ingredient, count in ingredient_counts.most_common(30):
    rank += 1
    print(f"{rank:02d}. {ingredient}: {count}")

## Task 2

Construct a recipe network from the data, based on the co-occurrence of pairs of ingredients in each recipe. Each node should be an ingredient, and an edge indicates that two ingredients occur at least once in the same recipe. Edge weights indicate the number of times the pair occur together.

Note: When building the network, exclude the following common items from the analysis:

In [ ]:
exclusions = ["water", "salt", "sugar", "butter", "pepper", "black pepper", "oil", "olive oil", "flour"]

In [ ]:
# firstly, count the ingredient pairs in each recipe:
pair_counts = Counter()
for recipe in data:
    # use itertools to find all unique combinations
    recipe_pairs = list( itertools.combinations(recipe["ingredients"], r=2) )
    for ingredient1, ingredient2 in recipe_pairs:
        # exclude this pair?
        if ingredient1 in exclusions or ingredient2 in exclusions:
            continue
        # convert to a set, because the order doesn't matter
        pair = frozenset([ingredient1, ingredient2])
        pair_counts[pair] += 1

In [ ]:
# now create the network from the pair frequencies:
g = nx.Graph()
for pair in pair_counts:
    g.add_edge( *pair, weight=pair_counts[pair] )
g.number_of_nodes(), g.number_of_edges()

## Task 3

Find the top ingredients in the recipe network, based on (i) degree centrality, (ii) weighted degree centrality. These measures will identify the most connected (i.e. important) ingredients in the network.

In [ ]:
# calculate the unweighted degree
degrees = dict(g.degree())
# convert to a Pandas series and sort it
sdeg = pd.Series(degrees, name="degree")
sdeg = sdeg.sort_values(ascending=False)
# display the top 10
pd.DataFrame(sdeg.head(10))

In [ ]:
# calculate the weighted degree
wdegrees = dict(g.degree(weight="weight"))
# convert to a Pandas series and sort it
swdeg = pd.Series(wdegrees, name="w-degree")
swdeg = swdeg.sort_values(ascending=False)
# display the top 10
pd.DataFrame(swdeg.head(10))

## Task 4

Filter the network to remove all edges with weight < 10 (i.e. all pairs of ingredients which have co-occurred in fewer than 10 recipes). This filtering step will focus the analysis on stronger ingredient associations.

Export this filtered network as a GEXF file for use in external network analysis tools.

In [ ]:
# create a new network
g2 = nx.Graph()
for node1, node2, attrs in g.edges(data=True):
    # add the edge?
    if attrs["weight"] >= 10:
        g2.add_edge( node1, node2, **attrs)
g2.number_of_nodes(), g2.number_of_edges()

In [ ]:
# export the filtered network
nx.write_gexf(g2, "recipes-filtered.gexf")

## Task 5

Load the GEXF file from Task 4 in *Gephi*, and complete the following steps:
    
1. Calculate the **degree** of the nodes
2. Colour and scale the size of the nodes, ranked based on their **degree**
3. Test different layout algorithms and parameter values to find a suitable layout for the network
4. Use the *Preview* screen to adjust the final appearance of the recipe network, and then export an image of the network as a PNG file

In [ ]:
# see recipes-filtered.gephi